# 2주차 8교시 · CSV 25개에서 API 응답까지 전체 흐름 다시 만들기

**학생용 지적실험노트**

오늘의 질문: **앞 교시의 핵심 빈칸만 보고 파일 찾기부터 API 확인까지 다시 연결할 수 있을까요?**

오늘의 완성 결과: CSV 25개를 30,000행으로 합치고 1,000행으로 요약한 뒤 36열로 변환해 API 응답까지 확인합니다.

## 시작하기 전에 폴더를 확인하세요

- `activity_batches`는 강사가 ZIP 안에 미리 넣어 준 폴더입니다. 학생이 새로 만들지 않습니다.
- ZIP을 먼저 압축 해제하고, 압축을 풀어서 생긴 최상위 폴더 전체를 VS Code에서 엽니다.
- 이 노트북이나 CSV 파일을 다른 폴더로 따로 옮기지 않습니다.
- 자세한 순서는 `week2/00_학생용_먼저읽기_압축해제와VSCode열기.md`를 확인합니다.

각 Q에서 ① 실행 전 예상 ② ___ 채우기 ③ 실행 ④ 값 하나 바꾸기 ⑤ 결과 이유 한 문장을 지킵니다. 오류 메시지에 ___가 보이면 고장이 아니라 아직 코드 빈칸이 남았다는 뜻입니다.


In [ ]:
from pathlib import Path
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
WEEK2_DIR = DATA_DIR.parents[1] / "week2"
OUTPUT_DIR = WEEK2_DIR / "outputs"
ARTIFACT_DIR = WEEK2_DIR / "artifacts"
OUTPUT_DIR.mkdir(exist_ok=True)
print("데이터 폴더:", DATA_DIR)


## Q1. CSV 25개의 위치를 찾아 종합실습 입력을 확정합니다

**먼저 예상하기**

- 처음부터 다시 시작할 때 가장 먼저 어떤 파일들을 사용할지 어떻게 확인할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 종합실습의 입력 범위를 스스로 확정합니다.

**현재 자료구조** batch_files는 Path 25개 list입니다.

**코드 읽기 도움** 정렬은 이후 결과와 오류 재현 순서를 고정합니다.

**정상 결과** activity_batch_01.csv activity_batch_25.csv

**오류가 나면** AssertionError. 폴더·패턴·첫 파일·마지막 파일을 차례로 출력합니다.


In [ ]:
batch_dir = DATA_DIR / ___
batch_files = sorted(batch_dir.glob(___))
assert len(batch_files) == ___
print(batch_files[0].name, batch_files[-1].name)

## Q2. 각 행에 출처를 남기며 CSV 25개를 30,000행으로 합칩니다

**먼저 예상하기**

- 합치기만 가능해도 source_file 열을 남겨야 하는 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 여러 파일 결합과 추적 가능성을 동시에 완성합니다.

**현재 자료구조** activity는 30,000행×10열 DataFrame입니다.

**코드 읽기 도움** for는 읽기와 출처 붙이기를 파일 25개에 똑같이 반복합니다.

**정상 결과** assert 통과, (30000, 10)

**오류가 나면** 행·열 불일치. frames 길이와 첫 DataFrame shape를 확인합니다.


In [ ]:
frames = []
for file_path in batch_files:
    one = pd.read_csv(file_path)
    one["source_file"] = ___
    frames.append(one)
activity = pd.___(frames, ignore_index=True)
assert activity.shape == ___

## Q3. 30일 활동을 회원별 한 행으로 요약합니다

**먼저 예상하기**

- 30,000행을 1,000행으로 줄인 것은 삭제일까요, 요약일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 원본 로그를 전처리기가 받을 수 있는 회원 단위 지표로 바꿉니다.

**현재 자료구조** member_activity는 1,000행×10열 DataFrame입니다.

**코드 읽기 도움** groupby는 member_id가 같은 행끼리 묶고 agg는 각 묶음에서 합계·평균·최댓값을 계산합니다.

**정상 결과** assert 통과, 1,000명

**오류가 나면** 1,000행이 아니거나 stay_time 변환 오류. member_id 고유값 1,000과 errors='coerce'를 확인합니다.


In [ ]:
activity["activity_dt"] = pd.to_datetime(activity["activity_date"], errors="coerce")
activity["stay_time_min_num"] = pd.to_numeric(activity["stay_time_min"], errors=___)
activity["active_dt"] = activity["activity_dt"].where(activity["logged_in"])
member_activity = activity.groupby("member_id").agg(
    active_days=("logged_in","sum"), total_app_opens=("app_opens","sum"),
    total_stay_time_min=("stay_time_min_num","sum"), total_profile_views=("profile_views","sum"),
    total_recommendations_viewed=("recommendations_viewed","sum"), total_messages_sent=("messages_sent","sum"),
    avg_reply_delay_sec=("avg_reply_delay_sec","mean"), last_active_date=("active_dt","max")
).reset_index()
member_activity["days_since_last_activity"] = (pd.Timestamp("2026-07-01") - member_activity["last_active_date"]).dt.days
assert len(member_activity) == ___

## Q4. 회원 기본정보를 붙이고 전처리할 13개 열을 고릅니다

**먼저 예상하기**

- 활동 요약과 회원정보를 붙인 뒤 어떤 열만 전처리기에 넣어야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 활동 지표와 나이·지역·회원권 같은 기본정보를 한 행에서 사용합니다.

**현재 자료구조** model_table은 1,000행×16열, X는 1,000행×13열 DataFrame입니다.

**코드 읽기 도움** left merge는 members 1,000명을 기준으로 유지합니다.

**정상 결과** (1000, 16) (1000, 13)

**오류가 나면** KeyError 또는 merge 행 증가. 열 이름과 양쪽 member_id 중복을 확인합니다.


In [ ]:
members = pd.read_csv(DATA_DIR / "members.csv")
member_columns = ["member_id","gender","age","residence_region","membership_tier","signup_channel","meeting_scheduled"]
model_table = members[member_columns].merge(member_activity, on="member_id", how=___, validate="one_to_one")
numeric_features = ["age","active_days","total_app_opens","total_stay_time_min","total_profile_views","total_recommendations_viewed","total_messages_sent","avg_reply_delay_sec","days_since_last_activity"]
categorical_features = ["gender","residence_region","membership_tier","signup_channel"]
X = model_table[___]
print(model_table.shape, X.shape)

## Q5. 저장한 전처리기로 1,000명을 36개 숫자 열로 바꿉니다

**먼저 예상하기**

- 종합실습에서 전처리 기준을 다시 배우지 않는 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 6교시에서 저장한 기준이 다른 실행에서도 그대로 재사용되는지 확인합니다.

**현재 자료구조** preprocessor는 joblib에서 불러온 객체, processed는 1,000행×36열 DataFrame입니다.

**코드 읽기 도움** load는 파일에 저장된 객체를 되살리고 transform은 저장된 기준만 적용합니다.

**정상 결과** (1000, 36) 0

**오류가 나면** FileNotFoundError 또는 NotFittedError. 6교시에서 만든 전처리 파일이 있는지 확인하고, 현재 코드의 경로와 실제 저장 위치가 같은지 살펴봅니다.


In [ ]:
import joblib
artifact_path = ARTIFACT_DIR / "matchmaking_preprocessor.joblib"
preprocessor = joblib.___(artifact_path)
processed = preprocessor.___(X)
assert processed.shape == ___
print(processed.shape, processed.isna().sum().sum())

## Q6. 요약표를 저장하고 API 두 주소의 응답을 검사합니다

**먼저 예상하기**

- 브라우저를 열지 않고도 API 응답을 확인할 수 있는 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 데이터 처리 결과가 실제 서버 응답까지 연결되는지 확인합니다.

**현재 자료구조** model_table은 CSV로 저장되고 TestClient 응답은 status_code와 JSON 데이터를 가집니다.

**코드 읽기 도움** TestClient는 실제 서버 포트를 열지 않고도 FastAPI 주소의 응답을 테스트합니다.

**정상 결과** {'status': 'ok', 'rows': 1000} 3

**오류가 나면** ModuleNotFoundError 또는 500. lesson7 경로와 api_app_answer.py, summary CSV 존재를 확인합니다.


In [ ]:
summary_path = OUTPUT_DIR / "member_activity_summary.csv"
model_table.to_csv(summary_path, index=False)

from fastapi.testclient import TestClient
import sys
lesson7_dir = WEEK2_DIR / "lesson7_fastapi"
sys.path.insert(0, str(lesson7_dir))
from api_app_answer import app
client = TestClient(app)
health = client.get(___)
members_response = client.get(___)
assert health.status_code == ___
assert len(members_response.json()) == 3
print(health.json(), len(members_response.json()))

## 마무리 확인

- [ ] 오늘 결과가 **CSV 25개를 30,000행으로 합치고 1,000행으로 요약한 뒤 36열로 변환해 API 응답까지 확인합니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
